In [ ]:
# Databricks notebook source
# MAGIC
# MAGIC **What we do:**
# MAGIC - Read raw transactions from Bronze
# MAGIC - Remove duplicates
# MAGIC - Handle nulls
# MAGIC - Standardize schema (rename columns, cast types)
# MAGIC - Add surrogate keys
# MAGIC - Add data quality flags
# MAGIC - Write to Silver Delta table (append-only)

In [ ]:
SP_CLIENT_ID     = dbutils.secrets.get(scope="retail-banking-scope", key="sp-client-id")
SP_TENANT_ID     = dbutils.secrets.get(scope="retail-banking-scope", key="sp-tenant-id")
SP_CLIENT_SECRET = dbutils.secrets.get(scope="retail-banking-scope", key="sp-client-secret")

STORAGE_ACCOUNT   = "retailbankingdl"
ADLS_BRONZE_PATH  = f"abfss://bronze@{STORAGE_ACCOUNT}.dfs.core.windows.net"
ADLS_SILVER_PATH  = f"abfss://silver@{STORAGE_ACCOUNT}.dfs.core.windows.net"

BRONZE_TABLE = f"{ADLS_BRONZE_PATH}/transactions_raw"
SILVER_TABLE = f"{ADLS_SILVER_PATH}/transactions_cleaned"

print("✅ Configuration loaded")

In [ ]:
spark.conf.set(
    f"fs.azure.account.auth.type.{STORAGE_ACCOUNT}.dfs.core.windows.net",
    "OAuth"
)
spark.conf.set(
    f"fs.azure.account.oauth.provider.type.{STORAGE_ACCOUNT}.dfs.core.windows.net",
    "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider"
)
spark.conf.set(
    f"fs.azure.account.oauth2.client.id.{STORAGE_ACCOUNT}.dfs.core.windows.net",
    SP_CLIENT_ID
)
spark.conf.set(
    f"fs.azure.account.oauth2.client.secret.{STORAGE_ACCOUNT}.dfs.core.windows.net",
    SP_CLIENT_SECRET
)
spark.conf.set(
    f"fs.azure.account.oauth2.client.endpoint.{STORAGE_ACCOUNT}.dfs.core.windows.net",
    f"https://login.microsoftonline.com/{SP_TENANT_ID}/oauth2/token"
)

print("✅ ADLS Gen2 connection configured")

In [ ]:
print("Reading from Bronze...")

df_bronze = spark.read.format("delta").load(BRONZE_TABLE)
bronze_count = df_bronze.count()

print(f"  ✅ Bronze rows loaded: {bronze_count:,}")
df_bronze.select("Time", "Amount", "Class", "source", "ingestion_date").show(5)

In [ ]:
print("Removing duplicates...")

# Deduplicate on all feature columns (V1-V28, Time, Amount, Class)
feature_cols = ["Time", "Amount", "Class"] + [f"V{i}" for i in range(1, 29)]

df_deduped = df_bronze.dropDuplicates(feature_cols)

duplicates_removed = bronze_count - df_deduped.count()
print(f"  ✅ Rows before: {bronze_count:,}")
print(f"  ✅ Duplicates removed: {duplicates_removed:,}")
print(f"  ✅ Rows after dedup: {df_deduped.count():,}")

In [ ]:
from pyspark.sql.functions import col, count, when, isnan

print("Checking for nulls...")

# Count nulls in key columns
null_counts = df_deduped.select([
    count(when(col(c).isNull() | isnan(c), c)).alias(c)
    for c in ["Time", "Amount", "Class"]
]).collect()[0]

print(f"  Nulls in Time:   {null_counts['Time']}")
print(f"  Nulls in Amount: {null_counts['Amount']}")
print(f"  Nulls in Class:  {null_counts['Class']}")

# Drop rows where critical columns are null
df_no_nulls = df_deduped.dropna(subset=["Time", "Amount", "Class"])

nulls_removed = df_deduped.count() - df_no_nulls.count()
print(f"\n  ✅ Null rows removed: {nulls_removed:,}")
print(f"  ✅ Rows remaining: {df_no_nulls.count():,}")

In [ ]:
from pyspark.sql.functions import col, lit, current_timestamp, sha2, concat_ws
from pyspark.sql.types import DoubleType, IntegerType, LongType
from datetime import datetime

print("Standardizing schema...")

# Rename and cast columns to standardized names
df_standardized = (df_no_nulls
    .withColumnRenamed("Time", "transaction_time_seconds")
    .withColumnRenamed("Amount", "transaction_amount")
    .withColumnRenamed("Class", "is_fraud")
    .withColumn("transaction_time_seconds", col("transaction_time_seconds").cast(DoubleType()))
    .withColumn("transaction_amount",       col("transaction_amount").cast(DoubleType()))
    .withColumn("is_fraud",                 col("is_fraud").cast(IntegerType()))
)

print(f"  ✅ Columns standardized")
print(f"  ✅ Types cast: transaction_time_seconds (Double), transaction_amount (Double), is_fraud (Integer)")

In [ ]:
from pyspark.sql.functions import sha2, concat_ws, when

print("Adding surrogate keys and DQ flags...")

ingestion_date = datetime.utcnow().strftime("%Y-%m-%d")

df_silver = (df_standardized
    # Surrogate key — hash of feature columns
    .withColumn(
        "transaction_sk",
        sha2(concat_ws("||",
            col("transaction_time_seconds").cast("string"),
            col("transaction_amount").cast("string"),
            col("V1").cast("string"),
            col("V2").cast("string")
        ), 256)
    )
    # Data quality flags
    .withColumn(
        "dq_amount_valid",
        when(col("transaction_amount") > 0, True).otherwise(False)
    )
    .withColumn(
        "dq_fraud_label_valid",
        when(col("is_fraud").isin(0, 1), True).otherwise(False)
    )
    .withColumn(
        "dq_passed",
        when(
            (col("dq_amount_valid") == True) &
            (col("dq_fraud_label_valid") == True),
            True
        ).otherwise(False)
    )
    # Silver audit columns
    .withColumn("silver_ingestion_ts",   current_timestamp())
    .withColumn("silver_ingestion_date", lit(ingestion_date))
    .withColumn("silver_source",         lit("bronze_transactions_raw"))
)

print(f"  ✅ Surrogate key added: transaction_sk")
print(f"  ✅ DQ flags added: dq_amount_valid, dq_fraud_label_valid, dq_passed")

In [ ]:
print("Data Quality Summary...\n")

total        = df_silver.count()
dq_passed    = df_silver.filter(col("dq_passed") == True).count()
dq_failed    = df_silver.filter(col("dq_passed") == False).count()
invalid_amt  = df_silver.filter(col("dq_amount_valid") == False).count()
invalid_label= df_silver.filter(col("dq_fraud_label_valid") == False).count()
fraud_rows   = df_silver.filter(col("is_fraud") == 1).count()
normal_rows  = df_silver.filter(col("is_fraud") == 0).count()

print(f"  Total rows:          {total:,}")
print(f"  DQ passed:           {dq_passed:,} ({round(dq_passed/total*100,2)}%)")
print(f"  DQ failed:           {dq_failed:,}")
print(f"  Invalid amounts:     {invalid_amt:,}")
print(f"  Invalid fraud label: {invalid_label:,}")
print(f"  Normal transactions: {normal_rows:,}")
print(f"  Fraud transactions:  {fraud_rows:,} ({round(fraud_rows/total*100,3)}%)")

In [ ]:
print(f"Writing to Silver Delta table...")
print(f"  Target: {SILVER_TABLE}")

(df_silver
    .write
    .format("delta")
    .mode("append")
    .partitionBy("silver_ingestion_date")
    .save(SILVER_TABLE)
)

print(f"✅ Silver table written successfully")

In [ ]:
print("Verifying Silver table...")

df_verify = spark.read.format("delta").load(SILVER_TABLE)
total_rows = df_verify.count()
partitions = df_verify.select("silver_ingestion_date").distinct().collect()

print(f"  ✅ Total rows: {total_rows:,}")
print(f"  ✅ Partitions: {[r['silver_ingestion_date'] for r in partitions]}")
print(f"  ✅ Columns: {len(df_verify.columns)}")

df_verify.select(
    "transaction_sk",
    "transaction_time_seconds",
    "transaction_amount",
    "is_fraud",
    "dq_passed",
    "silver_ingestion_date"
).show(5, truncate=True)

In [ ]:
print("=" * 60)
print("SILVER LAYER — TRANSACTIONS CLEANING COMPLETE")
print("=" * 60)
print(f"  ✅ Source: Bronze transactions_raw")
print(f"  ✅ Duplicates removed")
print(f"  ✅ Nulls handled")
print(f"  ✅ Schema standardized")
print(f"  ✅ Surrogate keys added")
print(f"  ✅ DQ flags added")
print(f"  ✅ Rows in Silver: {total_rows:,}")
print(f"  ✅ Location: {SILVER_TABLE}")
print("=" * 60)
print("Ready for Week 2 Day 3 — SCD Type 2 Customer Dimension")
print("=" * 60)